In [1]:
!pip install -q unsloth torch transformers flask pyngrok Pillow requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.7/192.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from unsloth import FastVisionModel
import torch
from PIL import Image
import io
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import TextStreamer

app = Flask(__name__)

# Load model
model, tokenizer = FastVisionModel.from_pretrained(
    "DucPTIT/Qwen2-VL-2B-VQA-Vietnamese-v2",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(model)

def process_vqa(image, question):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. You can only reply in Vietnamese"},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": question}
        ]}
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_prompt=True)
    output = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=128,
        use_cache=True,
        temperature=1.5,
        min_p=0.1
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

@app.route('/vqa', methods=['POST'])
def vqa():
    try:
        # Nhận hình ảnh và câu hỏi từ request
        image_file = request.files['image']
        question = request.form['question']
        image = Image.open(image_file)

        answer = process_vqa(image, question)
        return jsonify({'answer': answer})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Init ngrok to create URL public
!ngrok config add-authtoken 2vt4qc38uodVUFfag6ZWFk9iA0c_3vJUQEXJYaGCHtZ5sWFuW

public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")

# Run Flask server
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Qwen2_Vl patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

INFO:werkzeug:127.0.0.1 - - [26/Apr/2025 02:40:32] "POST /vqa HTTP/1.1" 200 -


Hình ảnh này mô tả một chú chó vàng đang đeo kính mát ở bãi biển trước khi mặt trời lên. Mặt trời đang ánh sáng vàng và ấm áp, tạo ra một ánh sáng vàng rực rỡ. Bãi biển và cây cọ xanh đang được ánh nắng mặt trời chiếu sáng.<|im_end|>


INFO:werkzeug:127.0.0.1 - - [26/Apr/2025 02:41:15] "POST /vqa HTTP/1.1" 200 -


Hãy giữ cho chú chó của bạn tránh nắng và hãy bảo vệ chúng khỏi các loại ánh sáng mạnh như mặt trời hoặc đèn led.<|im_end|>


INFO:werkzeug:127.0.0.1 - - [26/Apr/2025 02:43:12] "POST /vqa HTTP/1.1" 200 -
